<a href="https://colab.research.google.com/github/zainebelhaj/Citewise/blob/main/CiteWise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers openai numpy scikit-learn

# **Step 2: Creating the knowledge base (the documents)**

In [ ]:
from pathlib import Path

DOCS_DIR = Path("docs")
DOCS_DIR.mkdir(exist_ok=True)

documents = {
"01_annual_leave_policy.md": """# Annual Leave Policy
All full-time staff and interns receive 25 days of paid annual leave per year, plus UK bank holidays. Interns on the 9-month programme receive leave pro rata (about 1.5 days per month worked).
Leave requests must be submitted through the HR portal at least 2 weeks in advance for absences longer than 3 days. Requests of 3 days or fewer need 3 working days' notice.
Up to 5 unused days can be carried over into the next leave year. Carried-over days must be used by 31 March.
Leave over the final two weeks of December is allocated first-come-first-served, and managers confirm requests within 5 working days.
""",
"02_remote_working_policy.md": """# Remote and Hybrid Working Policy
Staff may work remotely up to 3 days per week. Team anchor days are Tuesday and Thursday, when everyone is expected in the Bradford office.
Interns should be in the office at least 3 days per week during the first 8 weeks to support onboarding.
Remote work must be done from within the UK unless approved in writing by a director. Working abroad is limited to 10 working days per year.
Employees must use the company VPN when accessing client data from home, and may not work on client data in public spaces such as cafes.
""",
"03_data_protection_and_security.md": """# Data Protection and Security Policy
All client data is classified as Confidential. Confidential data must not be pasted into public generative AI tools such as free chatbots.
Approved AI tools for client work are limited to the company-hosted LLM gateway and approved enterprise accounts. Using any other tool requires approval from the Data Protection Officer (DPO).
Personal data must be anonymised or pseudonymised before use in model training or evaluation.
A suspected data breach must be reported to the DPO within 24 hours. The DPO decides whether the ICO must be notified within 72 hours.
Laptops must use full-disk encryption and multi-factor authentication. Passwords must be at least 14 characters.
""",
"04_ai_use_and_governance.md": """# Responsible AI Use and Governance
Every AI feature delivered to a client must have a documented model card covering intended use, training data sources, known limitations and evaluation results.
Generative AI outputs used in client deliverables must be reviewed by a human before release. Reviewers record their sign-off in the project log.
Retrieval-augmented systems must show source citations and must decline to answer when no supporting evidence is found.
Staff must disclose use of generative AI in their own work. Disclosure includes the tool used, the purpose and how the output was checked.
Bias testing is required for any model that affects individuals, for example screening or triage models. Results are reviewed by the AI Governance Group each quarter.
""",
"05_expenses_and_travel.md": """# Expenses and Travel Policy
Claim expenses within 30 days through the finance portal, attaching receipts for any item over 10 pounds.
Rail travel must be standard class and booked at least 7 days in advance where possible. Air travel needs director approval.
Hotel stays are capped at 120 pounds per night outside London and 160 pounds per night in London.
The meal allowance when travelling is 25 pounds per day. Alcohol is not reimbursable.
Mileage is reimbursed at 45 pence per mile for the first 10,000 business miles each tax year.
""",
"06_internship_programme_handbook.md": """# Internship Programme Handbook
The internship runs for 9 months. Interns work on one live client project and complete a professional portfolio alongside it.
Each intern has a technical mentor and a line manager. One-to-one meetings with the mentor happen weekly for the first 3 months, then every two weeks.
Formal progress reviews take place at months 3, 6 and 9. The month 6 review includes a mock interview with a panel.
Interns are expected to log 4 hours per week of structured learning time. Approved learning includes courses, certifications and internal workshops.
Interns who complete the programme are eligible to apply for graduate roles at FAI4ALL. There is no guaranteed offer.
""",
"07_incident_response_runbook.md": """# Production Incident Response Runbook
Severity 1 means a client-facing outage or data exposure. Severity 2 means major degradation with a workaround. Severity 3 means a minor issue.
For Severity 1, page the on-call engineer immediately and open an incident channel. The on-call engineer must acknowledge within 15 minutes.
The incident commander posts an update every 30 minutes until the incident is resolved.
After resolution, a blameless post-incident review must be written within 5 working days. It includes the timeline, root cause, impact and follow-up actions.
Model rollbacks use the previous tagged model version in the registry. Rollbacks must be logged in the project log.
""",
"08_llm_gateway_and_cost_controls.md": """# LLM Gateway and Cost Controls
All LLM calls from client projects go through the company LLM gateway, which logs prompts, responses, latency and token cost.
Each project has a monthly LLM budget. The default is 500 pounds. Alerts trigger at 80 percent of budget and calls are throttled at 100 percent.
Use the smallest model that meets the quality target. Larger models need a written justification in the design document.
Embedding vectors are cached to avoid repeated cost. Responses to identical queries may be cached for up to 24 hours.
Build-versus-buy decisions for new AI components must compare total cost of ownership over 12 months.
""",
}

for name, text in documents.items():
    (DOCS_DIR / name).write_text(text, encoding="utf-8")

print(f"Created {len(documents)} documents in ./{DOCS_DIR}/")
for p in sorted(DOCS_DIR.glob("*.md")):
    print(" -", p.name, f"({len(p.read_text().split())} words)")

Created 8 documents in ./docs/
 - 01_annual_leave_policy.md (110 words)
 - 02_remote_working_policy.md (102 words)
 - 03_data_protection_and_security.md (111 words)
 - 04_ai_use_and_governance.md (120 words)
 - 05_expenses_and_travel.md (92 words)
 - 06_internship_programme_handbook.md (112 words)
 - 07_incident_response_runbook.md (105 words)
 - 08_llm_gateway_and_cost_controls.md (106 words)


# **Step 3: Spliting the documents into chunks**

In [ ]:
def load_chunks(docs_dir="docs", window=2):
    chunks = []
    for path in sorted(Path(docs_dir).glob("*.md")):
        lines = [ln.strip() for ln in path.read_text(encoding="utf-8").splitlines() if ln.strip()]
        title = lines[0].lstrip("# ").strip()   # first line is the heading
        body = lines[1:]                        # the rest are the facts
        for i in range(0, max(1, len(body) - window + 1)):
            piece = " ".join(body[i : i + window])
            chunks.append({"source": path.name, "title": title, "text": piece})
    return chunks

chunks = load_chunks()
print("Total chunks:", len(chunks))
print()
for c in chunks[:3]:
    print(c["source"], "|", c["title"])
    print(c["text"])
    print()

Total chunks: 30

01_annual_leave_policy.md | Annual Leave Policy
All full-time staff and interns receive 25 days of paid annual leave per year, plus UK bank holidays. Interns on the 9-month programme receive leave pro rata (about 1.5 days per month worked). Leave requests must be submitted through the HR portal at least 2 weeks in advance for absences longer than 3 days. Requests of 3 days or fewer need 3 working days' notice.

01_annual_leave_policy.md | Annual Leave Policy
Leave requests must be submitted through the HR portal at least 2 weeks in advance for absences longer than 3 days. Requests of 3 days or fewer need 3 working days' notice. Up to 5 unused days can be carried over into the next leave year. Carried-over days must be used by 31 March.

01_annual_leave_policy.md | Annual Leave Policy
Up to 5 unused days can be carried over into the next leave year. Carried-over days must be used by 31 March. Leave over the final two weeks of December is allocated first-come-first-serv

# **Step 4: Turning the chunks into embeddings**

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [f"{c['title']}. {c['text']}" for c in chunks]
vectors = model.encode(texts, normalize_embeddings=True)

print("Vectors shape:", vectors.shape)
print("First 8 numbers of chunk 0:", np.round(vectors[0][:8], 3))

question = "How many days of holiday do I get?"
q_vec = model.encode([question], normalize_embeddings=True)
scores = (vectors @ q_vec.T).ravel()     # .ravel() flattens it to a simple list of 30 scores
best = int(np.argmax(scores))

print()
print("Question:", question)
print("Best match score:", round(float(scores[best]), 3))
print("Best chunk:", chunks[best]["text"][:150], "...")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectors shape: (30, 384)
First 8 numbers of chunk 0: [ 0.078 -0.032  0.018 -0.062  0.022  0.007 -0.014 -0.094]

Question: How many days of holiday do I get?
Best match score: 0.437
Best chunk: Up to 5 unused days can be carried over into the next leave year. Carried-over days must be used by 31 March. Leave over the final two weeks of Decemb ...


# **Step 5: Building the retrieval function and the “no evidence” guardrail**

In [ ]:
TOP_K = 3
THRESHOLD = 0.35   # a first guess; we will tune it later using evaluation

def retrieve(question, k=TOP_K):
    q_vec = model.encode([question], normalize_embeddings=True)
    scores = (vectors @ q_vec.T).ravel()
    top = np.argsort(-scores)[:k]               # indexes of the k highest scores
    return [(chunks[i], float(scores[i])) for i in top]

def check_evidence(question):
    hits = retrieve(question)
    best_score = hits[0][1]
    enough = best_score >= THRESHOLD
    print("Q:", question)
    print(f"Best score: {best_score:.3f} ->", "ENOUGH evidence" if enough else "NO evidence (would decline)")
    for n, (chunk, score) in enumerate(hits, 1):
        print(f"  [{n}] {score:.3f} | {chunk['source']} | {chunk['text'][:90]}...")
    print()

check_evidence("How many days of holiday do I get?")
check_evidence("Can I paste client data into a free chatbot?")
check_evidence("What is the hotel cap in London?")
check_evidence("What is the company pension contribution?")
check_evidence("Who won the football match last night?")

Q: How many days of holiday do I get?
Best score: 0.437 -> ENOUGH evidence
  [1] 0.437 | 01_annual_leave_policy.md | Up to 5 unused days can be carried over into the next leave year. Carried-over days must b...
  [2] 0.337 | 01_annual_leave_policy.md | Leave requests must be submitted through the HR portal at least 2 weeks in advance for abs...
  [3] 0.330 | 01_annual_leave_policy.md | All full-time staff and interns receive 25 days of paid annual leave per year, plus UK ban...

Q: Can I paste client data into a free chatbot?
Best score: 0.479 -> ENOUGH evidence
  [1] 0.479 | 03_data_protection_and_security.md | All client data is classified as Confidential. Confidential data must not be pasted into p...
  [2] 0.257 | 03_data_protection_and_security.md | Approved AI tools for client work are limited to the company-hosted LLM gateway and approv...
  [3] 0.205 | 04_ai_use_and_governance.md | Generative AI outputs used in client deliverables must be reviewed by a human before relea...

Q:

# **Step 6: Connecting to an LLM (the “G” in RAG)**

In [ ]:
from google.colab import userdata
from openai import OpenAI

LLM_API_KEY = userdata.get("LLM_API_KEY")
LLM_BASE_URL = "https://api.groq.com/openai/v1"

client = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)

# Ask the provider which models are available right now
models = sorted(m.id for m in client.models.list().data)
print("Connected. Available models:")
for m in models:
    print(" -", m)

Connected. Available models:
 - allam-2-7b
 - canopylabs/orpheus-arabic-saudi
 - canopylabs/orpheus-v1-english
 - meta-llama/llama-prompt-guard-2-22m
 - meta-llama/llama-prompt-guard-2-86m
 - openai/gpt-oss-120b
 - openai/gpt-oss-20b
 - openai/gpt-oss-safeguard-20b
 - qwen/qwen3.8-27b
 - whisper-large-v3
 - whisper-large-v3-turbo


# **Step 7: Building the full ask() function (retrieve, guard, generate, cite)**

In [ ]:
import time

MODEL = "openai/gpt-oss-20b"

# Approximate prices in GBP per 1M tokens. Check Groq's pricing page and adjust.
PRICE_IN_PER_1M = 0.06
PRICE_OUT_PER_1M = 0.24

SYSTEM_PROMPT = (
    "You are a company knowledge assistant. Answer ONLY using the numbered context passages. "
    "Cite the passages you used like [1] or [2]. Keep the answer short. "
    "If the context does not contain the answer, reply exactly: "
    "I don't have enough evidence in the documents to answer that."
)

def ask(question):
    start = time.perf_counter()
    hits = retrieve(question)
    best_score = hits[0][1]

    # Guardrail: no evidence -> no LLM call, no cost
    if best_score < THRESHOLD:
        return {"question": question,
                "answer": "I don't have enough evidence in the documents to answer that.",
                "declined": True, "best_score": best_score, "hits": hits,
                "latency_s": time.perf_counter() - start, "tokens": 0, "cost_gbp": 0.0}

    context = "\n".join(f"[{i}] ({c['title']}) {c['text']}" for i, (c, s) in enumerate(hits, 1))
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        extra_body={"reasoning_effort": "low"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    answer = resp.choices[0].message.content.strip()
    u = resp.usage
    cost = u.prompt_tokens / 1e6 * PRICE_IN_PER_1M + u.completion_tokens / 1e6 * PRICE_OUT_PER_1M
    return {"question": question, "answer": answer,
            "declined": answer.lower().startswith("i don't have enough evidence"),
            "best_score": best_score, "hits": hits,
            "latency_s": time.perf_counter() - start,
            "tokens": u.prompt_tokens + u.completion_tokens, "cost_gbp": cost}

def show(question):
    r = ask(question)
    print("Q:", r["question"])
    print("A:", r["answer"])
    print(f"   [declined={r['declined']} | score={r['best_score']:.2f} | "
          f"{r['latency_s']:.1f}s | {r['tokens']} tokens | £{r['cost_gbp']:.6f}]")
    for i, (c, s) in enumerate(r["hits"], 1):
        print(f"   source [{i}] {c['source']} ({s:.2f})")
    print()

show("How many days of holiday do I get?")
show("Can I paste client data into a free chatbot?")
show("What is the company pension contribution?")

Q: How many days of holiday do I get?
A: You receive 25 days of paid annual leave per year, plus UK bank holidays. [3]
   [declined=False | score=0.44 | 0.5s | 465 tokens | £0.000044]
   source [1] 01_annual_leave_policy.md (0.44)
   source [2] 01_annual_leave_policy.md (0.34)
   source [3] 01_annual_leave_policy.md (0.33)

Q: Can I paste client data into a free chatbot?
A: I don't have enough evidence in the documents to answer that.
   [declined=True | score=0.48 | 0.3s | 377 tokens | £0.000029]
   source [1] 03_data_protection_and_security.md (0.48)
   source [2] 03_data_protection_and_security.md (0.26)
   source [3] 04_ai_use_and_governance.md (0.21)

Q: What is the company pension contribution?
A: I don't have enough evidence in the documents to answer that.
   [declined=True | score=0.27 | 0.0s | 0 tokens | £0.000000]
   source [1] 01_annual_leave_policy.md (0.27)
   source [2] 08_llm_gateway_and_cost_controls.md (0.17)
   source [3] 05_expenses_and_travel.md (0.16)



In [ ]:
SYSTEM_PROMPT = (
    "You are a company knowledge assistant. Answer using ONLY the numbered context passages. "
    "If a passage answers the question, even indirectly (for example a rule that implies a yes or no), "
    "answer it clearly and cite it like [1] or [2]. Keep the answer short. "
    "Only if NONE of the passages is relevant to the question, reply exactly: "
    "I don't have enough evidence in the documents to answer that."
)

show("Can I paste client data into a free chatbot?")
show("How many days of holiday do I get?")
show("What is the company pension contribution?")

Q: Can I paste client data into a free chatbot?
A: No. According to the policy, client data must not be pasted into public generative AI tools such as free chatbots. [1]
   [declined=False | score=0.48 | 0.6s | 440 tokens | £0.000040]
   source [1] 03_data_protection_and_security.md (0.48)
   source [2] 03_data_protection_and_security.md (0.26)
   source [3] 04_ai_use_and_governance.md (0.21)

Q: How many days of holiday do I get?
A: You receive 25 days of paid annual leave per year, plus UK bank holidays. [3]
   [declined=False | score=0.44 | 0.4s | 478 tokens | £0.000043]
   source [1] 01_annual_leave_policy.md (0.44)
   source [2] 01_annual_leave_policy.md (0.34)
   source [3] 01_annual_leave_policy.md (0.33)

Q: What is the company pension contribution?
A: I don't have enough evidence in the documents to answer that.
   [declined=True | score=0.27 | 0.0s | 0 tokens | £0.000000]
   source [1] 01_annual_leave_policy.md (0.27)
   source [2] 08_llm_gateway_and_cost_controls.md (0.17)
 

# **Step 8: Evaluating retrieval and the guardrail**

In [ ]:
# (question, expected source file or None if the documents do NOT contain the answer)
eval_set = [
    ("How many days of annual leave do staff get?", "01_annual_leave_policy.md"),
    ("How many unused leave days can I carry over?", "01_annual_leave_policy.md"),
    ("How many days a week can I work from home?", "02_remote_working_policy.md"),
    ("Can I work from abroad?", "02_remote_working_policy.md"),
    ("Can I paste client data into a free chatbot?", "03_data_protection_and_security.md"),
    ("How soon must a data breach be reported to the DPO?", "03_data_protection_and_security.md"),
    ("What must a model card include?", "04_ai_use_and_governance.md"),
    ("Must RAG systems show citations?", "04_ai_use_and_governance.md"),
    ("What is the hotel cap in London?", "05_expenses_and_travel.md"),
    ("What is the mileage rate?", "05_expenses_and_travel.md"),
    ("How long is the internship programme?", "06_internship_programme_handbook.md"),
    ("When is the mock interview in the internship?", "06_internship_programme_handbook.md"),
    ("How quickly must on-call acknowledge a Severity 1 incident?", "07_incident_response_runbook.md"),
    ("How soon must a post-incident review be written?", "07_incident_response_runbook.md"),
    ("What is the default monthly LLM budget per project?", "08_llm_gateway_and_cost_controls.md"),
    ("What happens at 80 percent of the LLM budget?", "08_llm_gateway_and_cost_controls.md"),
    # Questions the documents CANNOT answer (the guardrail should decline these)
    ("What is the company pension contribution?", None),
    ("Who won the football match last night?", None),
    ("What is the dress code in the office?", None),
    ("What is the salary of the CEO?", None),
    ("What is the capital of France?", None),
    ("How do I book a meeting room?", None),
]

answerable = [(q, s) for q, s in eval_set if s is not None]
unanswerable = [(q, s) for q, s in eval_set if s is None]

# --- Retrieval quality: is the right document in the top 3?
top1 = top3 = 0
print("RETRIEVAL (answerable questions)")
for q, expected in answerable:
    hits = retrieve(q)
    sources = [c["source"] for c, _ in hits]
    top1 += sources[0] == expected
    top3 += expected in sources
    if expected not in sources:
        print("  MISS:", q)
print(f"  Top-1 correct: {top1}/{len(answerable)}   Top-3 correct: {top3}/{len(answerable)}\n")

# --- Guardrail: sweep the threshold
ans_scores = [retrieve(q)[0][1] for q, _ in answerable]
una_scores = [retrieve(q)[0][1] for q, _ in unanswerable]
print(f"Best-score range  answerable: {min(ans_scores):.2f} to {max(ans_scores):.2f}")
print(f"Best-score range  unanswerable: {min(una_scores):.2f} to {max(una_scores):.2f}\n")

print("GUARDRAIL THRESHOLD SWEEP")
print("threshold | answered correctly | declined correctly")
for t in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45]:
    a = sum(s >= t for s in ans_scores) / len(ans_scores)
    d = sum(s < t for s in una_scores) / len(una_scores)
    print(f"   {t:.2f}   |       {a:4.0%}        |       {d:4.0%}")

RETRIEVAL (answerable questions)
  Top-1 correct: 15/16   Top-3 correct: 16/16

Best-score range  answerable: 0.21 to 0.81
Best-score range  unanswerable: 0.07 to 0.27

GUARDRAIL THRESHOLD SWEEP
threshold | answered correctly | declined correctly
   0.15   |       100%        |        33%
   0.20   |       100%        |        67%
   0.25   |        94%        |        67%
   0.30   |        88%        |       100%
   0.35   |        88%        |       100%
   0.40   |        88%        |       100%
   0.45   |        88%        |       100%


# **Step 9: Setting the threshold and find the weak spots**

In [ ]:
THRESHOLD = 0.30   # chosen from the sweep in Step 8

print("Answerable questions the guardrail would WRONGLY decline (score below threshold):")
for q, expected in answerable:
    hits = retrieve(q)
    score = hits[0][1]
    if score < THRESHOLD:
        print(f"  {score:.2f} | {q}")
        print(f"         best chunk: {hits[0][0]['text'][:100]}...")

print("\nAnswerable questions where the top-1 chunk was from the WRONG document:")
for q, expected in answerable:
    hits = retrieve(q)
    if hits[0][0]["source"] != expected:
        print(f"  {q}")
        print(f"         got {hits[0][0]['source']} instead of {expected}")

print("\nUnanswerable questions with the highest scores (closest to slipping through):")
ranked = sorted(((retrieve(q)[0][1], q) for q, _ in unanswerable), reverse=True)
for score, q in ranked[:3]:
    print(f"  {score:.2f} | {q}")

Answerable questions the guardrail would WRONGLY decline (score below threshold):
  0.27 | What must a model card include?
         best chunk: Use the smallest model that meets the quality target. Larger models need a written justification in ...
  0.21 | Must RAG systems show citations?
         best chunk: Retrieval-augmented systems must show source citations and must decline to answer when no supporting...

Answerable questions where the top-1 chunk was from the WRONG document:
  What must a model card include?
         got 08_llm_gateway_and_cost_controls.md instead of 04_ai_use_and_governance.md

Unanswerable questions with the highest scores (closest to slipping through):
  0.27 | What is the company pension contribution?
  0.25 | How do I book a meeting room?
  0.20 | What is the salary of the CEO?


# **Step 10: the full end-to-end test**

In [ ]:
# Each answerable question gets a keyword that a correct answer should contain
checks = {
    "How many days of annual leave do staff get?": "25",
    "How many unused leave days can I carry over?": "5",
    "How many days a week can I work from home?": "3",
    "Can I work from abroad?": "approv",
    "Can I paste client data into a free chatbot?": "not",
    "How soon must a data breach be reported to the DPO?": "24",
    "What must a model card include?": "intended use",
    "Must RAG systems show citations?": "cit",
    "What is the hotel cap in London?": "160",
    "What is the mileage rate?": "45",
    "How long is the internship programme?": "9",
    "When is the mock interview in the internship?": "6",
    "How quickly must on-call acknowledge a Severity 1 incident?": "15",
    "How soon must a post-incident review be written?": "5",
    "What is the default monthly LLM budget per project?": "500",
    "What happens at 80 percent of the LLM budget?": "alert",
}

results = []
for q, expected_source in eval_set:
    r = ask(q)
    if expected_source is not None:      # answerable: should answer AND contain the key fact
        ok = (not r["declined"]) and checks[q].lower() in r["answer"].lower()
    else:                                # unanswerable: should decline
        ok = r["declined"]
    results.append((ok, expected_source is not None, r))
    print(("PASS" if ok else "FAIL"), "|", q)
    print("      ", r["answer"][:110])

ans = [x for x in results if x[1]]
una = [x for x in results if not x[1]]
print("\n===== SUMMARY =====")
print(f"Answerable questions answered correctly: {sum(x[0] for x in ans)}/{len(ans)}")
print(f"Unanswerable questions correctly declined: {sum(x[0] for x in una)}/{len(una)}")
print(f"Average latency: {sum(x[2]['latency_s'] for x in results)/len(results):.2f}s")
print(f"Total tokens: {sum(x[2]['tokens'] for x in results)}")
print(f"Total estimated cost: £{sum(x[2]['cost_gbp'] for x in results):.5f}")

PASS | How many days of annual leave do staff get?
       Staff receive 25 days of paid annual leave per year. [2]
PASS | How many unused leave days can I carry over?
       Up to 5 unused days can be carried over into the next leave year. [1]
PASS | How many days a week can I work from home?
       You can work from home up to 3 days per week. [2]
PASS | Can I work from abroad?
       Yes, but only up to 10 working days per year unless you obtain written approval from a director. [1] [2]
PASS | Can I paste client data into a free chatbot?
       No. According to the policy, client data must not be pasted into public generative AI tools such as free chatb
PASS | How soon must a data breach be reported to the DPO?
       A data breach must be reported to the DPO within 24 hours. [1]
FAIL | What must a model card include?
       I don't have enough evidence in the documents to answer that.
FAIL | Must RAG systems show citations?
       I don't have enough evidence in the documents to ans

# **Step 11: A chat box for your demo**

In [13]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

box = widgets.Text(placeholder="Ask a question about the company documents...",
                   layout=widgets.Layout(width="75%"))
button = widgets.Button(description="Ask", button_style="primary")
output = widgets.Output()

def on_click(_):
    question = box.value.strip()
    if not question:
        return
    with output:
        clear_output()
        r = ask(question)
        status = " **Declined: no evidence in the documents**" if r["declined"] else " **Answered from the documents**"
        md = f"### {r['question']}\n\n{r['answer']}\n\n{status}  \n"
        md += f"Top match score: `{r['best_score']:.2f}` · Time: `{r['latency_s']:.1f}s` · "
        md += f"Tokens: `{r['tokens']}` · Est. cost: `£{r['cost_gbp']:.6f}`\n\n**Sources**\n\n"
        for i, (c, s) in enumerate(r["hits"], 1):
            md += f"> **[{i}] {c['title']}** (`{c['source']}`, score {s:.2f})  \n> {c['text']}\n\n"
        display(Markdown(md))

button.on_click(on_click)
display(widgets.HBox([box, button]), output)

Output()